# Chorus on Colab

Runs the echo-chamber simulation against a self-hosted open-weight model.
Cost is GPU time only; the model weights are free.

**Set the runtime to a GPU first:** Runtime > Change runtime type > T4 GPU.

Two things about notebooks that matter here:

- `asyncio.run()` cannot be called from inside a running event loop, and a
  notebook always has one. So the run is launched as a **subprocess**
  (`!python run_simulation.py`), not imported into a cell.
- The Colab filesystem is **ephemeral**. Cell 2 mounts Drive so a run that
  outlives a session is not lost. Skip it only for a throwaway smoke test.


## 1. Check the GPU


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

# vLLM needs compute capability >= 7.0.
#   T4  = 7.5  OK (no bfloat16, so --dtype half is required)
#   L4  = 8.9  OK
#   A100= 8.0  OK
#   P100= 6.0  NOT SUPPORTED, vLLM will refuse to start


## 2. Persist output to Drive (recommended)

Skip this only if you do not mind losing the run when the session ends.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RUNS_DIR = '/content/drive/MyDrive/chorus_runs'
!mkdir -p "$RUNS_DIR"
print('output ->', RUNS_DIR)


## 3. Install and clone

vLLM is a large install; this takes a few minutes.


In [ ]:
# Colab preinstalls a CUDA 12.x torch stack. vLLM upgrades torch to a CUDA 13
# build but leaves torchaudio behind, and transformers imports torchaudio,
# which then refuses to load against a mismatched torch:
#   RuntimeError: PyTorch and TorchAudio were compiled with different CUDA versions
#
# Remove ONLY torchaudio. Do NOT remove torchvision: vLLM imports it for its
# multimodal registry during engine startup, and without it the engine core
# dies with the unhelpful 'Engine core initialization failed'.
!pip uninstall -q -y torchaudio

!pip install -q vllm
!git clone -q https://github.com/ResearchDrafts/Chorus--Agentic-Network-Orchestration-and-Analysis.git
%cd Chorus--Agentic-Network-Orchestration-and-Analysis
!pip install -q -e .

# The red 'pip dependency resolver' wall above is Colab's preinstalled RAPIDS
# packages losing their CUDA 12 pins. None of them are used here.
print('installed. RESTART THE RUNTIME NOW, then run from the next cell.')


## 3b. Restart, then verify

The install upgrades torch, so the session must restart before vLLM can
import cleanly. **Runtime > Restart session**, then run the cell below.


In [ ]:
# Run this AFTER restarting. If it prints OK, vLLM can start.
import torch, importlib.util
print('torch      :', torch.__version__, '| CUDA', torch.version.cuda)
print('torchaudio :', 'absent (good)' if importlib.util.find_spec('torchaudio') is None else 'present')
import vllm
print('vllm       :', vllm.__version__)
print('GPU        :', torch.cuda.get_device_name(0))
print('\nOK, vLLM imports cleanly.')


## 4. Pick a model that fits the GPU

Weights at FP16, before KV cache and activations:

| Model | Weights | Fits 16 GB T4? |
|---|---|---|
| `Qwen2.5-3B-Instruct` | ~6 GB | yes, comfortably |
| `Qwen2.5-7B-Instruct` | ~15 GB | only with `--max-model-len` trimmed |
| `Qwen2.5-VL-7B-Instruct` | ~18 GB | no, needs L4/A100 or 2 GPUs |

`--max-model-len 4096` matters: prompts here run 400-800 tokens, so the
default 131K context reserves KV cache you will never use.


In [ ]:
import torch

gb = torch.cuda.get_device_properties(0).total_memory / 1e9
cc = torch.cuda.get_device_capability(0)

# 7B is ~15 GB of weights at FP16, which does not leave room for KV cache
# on a 16 GB T4 but is comfortable on a 24 GB L4 or A100.
MODEL = 'Qwen/Qwen2.5-7B-Instruct' if gb > 20 else 'Qwen/Qwen2.5-3B-Instruct'

# bfloat16 needs compute capability 8.0. Below that (T4 = 7.5) vLLM must be
# forced to fp16; at or above it, 'auto' picks the model's native dtype,
# which for Qwen is bfloat16 and numerically better than fp16.
DTYPE = 'auto' if cc >= (8, 0) else 'half'

print(f'{torch.cuda.get_device_name(0)}  {gb:.0f} GB  compute capability {cc[0]}.{cc[1]}')
print(f'  model {MODEL}')
print(f'  dtype {DTYPE}')


## 5. Start the vLLM server

Runs in the background. The wait loop below blocks until it is actually
serving, which takes a couple of minutes on first run while weights download.


In [ ]:
import subprocess, time, urllib.request, json, os, torch

# --enforce-eager: vLLM crashes during 'Capturing CUDA graphs' on Colab's
#   stack (Python 3.13 + CUDA 13), dying with the unhelpful 'Engine core
#   initialization failed'. Graph capture is a throughput optimisation only,
#   worth maybe 10-20%, so skipping it costs far less than a server that
#   will not stay up.
# --max-num-seqs 32: the orchestrator dispatches all M agents at once. At
#   M=100 that is ~23 GB of KV cache at 4096 context, more than an L4 has
#   free after weights. vLLM queues the rest rather than thrashing.
# --max-model-len 2048: prompts run ~800-1000 tokens plus ~150 completion,
#   so 4096 reserved double the KV cache actually needed.
args = ['vllm', 'serve', MODEL, '--dtype', DTYPE,
        '--enforce-eager',
        '--max-model-len', '2048',
        '--max-num-seqs', '32',
        '--gpu-memory-utilization', '0.90',
        '--port', '8000']

server = subprocess.Popen(args, stdout=open('/content/vllm.log','w'),
                          stderr=subprocess.STDOUT)
print(' '.join(args))

for attempt in range(180):  # up to ~15 minutes
    try:
        with urllib.request.urlopen('http://localhost:8000/v1/models', timeout=2) as r:
            print('server up:', json.loads(r.read())['data'][0]['id'])
            break
    except Exception:
        if server.poll() is not None:
            log = open('/content/vllm.log').read()
            print('server died. root cause:')
            for marker in ('Capturing CUDA graphs', 'Traceback (most recent call last)'):
                i = log.find(marker)
                if i > 0:
                    print(log[i:i+3000]); break
            else:
                print(log[-3000:])
            break
        time.sleep(5)
else:
    print('timed out; check /content/vllm.log')


## 6. Write a config

`model_backend_id` must match the served name exactly, prefixed with
`hosted_vllm/`. `api_base` is required: without it litellm routes on the
prefix alone and never reaches localhost.

Start small. M=5, K=1 is five calls and proves the whole path.


In [ ]:
import yaml, pathlib

cfg = {
    'run_id': 'colab_smoke',
    'rq_target': 'RQ1_RQ2',
    'topic': 'whether remote work should be the default for office jobs',
    'alpha': 0.5,
    'M': 5, 'N': 2, 'K': 1,
    'trial_number': 1,
    'language_condition': 'english',
    'model_backend_id': f'hosted_vllm/{MODEL}',
    'api_base': 'http://localhost:8000/v1',
    'stance_scale': [1, 2, 3, 4, 5, 6, 7],
    'stance_low_label': 'remote work should never be the default',
    'stance_high_label': 'remote work should always be the default',
    'persona_pool_id': 'test_pool',
    'meme_injection': {'enabled': False},
    'seed': 42,
    'temperature': 0.7,
    'rate_limits': {},          # empty: let vLLM batch all agents
    'checkpoint_every_n_turns': 1,
}
pathlib.Path('configs/colab_smoke.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False))
!python run_simulation.py configs/colab_smoke.yaml --dry-run


## 7. Run it

Subprocess, not an import: a notebook already has an event loop, so
`asyncio.run()` inside a cell would raise.


In [ ]:
RUNS = RUNS_DIR if 'RUNS_DIR' in dir() else 'runs'
!python run_simulation.py configs/colab_smoke.yaml --runs-dir "$RUNS"


## 8. Look at what came out

Three things decide whether this is worth scaling up.


In [ ]:
import pandas as pd, json, os

df = pd.read_json(f'{RUNS}/colab_smoke/interactions.jsonl', lines=True)
print(df[['speaker_agent_id','turn','stance_before','stance_after','api_call_status']])
print()
print('1. all calls succeeded :', (df.api_call_status == 'success').all())
print('2. stances moved       :', (df.stance_before != df.stance_after).any())
print('3. mean prompt tokens  :', int(df.prompt_token_count.mean()))
print()
print('--- one agent\'s reasoning ---')
print(df.reason_text.iloc[0][:400])


### Reading that

- **`failed_logged_null` rows** mean the model is not emitting the
  `STANCE: <n>` line. Some of this is normal from a 3B model. A lot of it
  means the model is too weak for this task, which matters most for
  Hinglish, where formatting is hardest.
- **Stances never moving** points at alpha or the prompt, not the model.
- **`reason_text`** should sound like the persona and address the topic.

## 9. Scaling up

Raise `M`, `K`, and `trial_number`, then sweep `language_condition` across
`english` and `hinglish` with the **same seed**, so neighbour sampling is
held constant and only language varies.

Before a real campaign, check Hinglish quality on ~100 calls. If the model
cannot produce consistent Hinglish, RQ1 measures model deficiency rather
than a language effect, and no analysis fixes that afterwards.

A run interrupted by a session timeout resumes byte-identically from its
checkpoint: re-run the same command, provided `--runs-dir` pointed at Drive.


## Shutting down


In [ ]:
server.terminate()
print('server stopped')


---
# Part 2: the real campaign

Everything above was a smoke test. This section runs the actual RQ1/RQ2
grid and stores the data.

**Design, and what is deliberately NOT here.**

- **4 topics x 2 languages**, one `ExperimentRun` per cell. Topic is an
  across-run dimension, not a within-run one, because `ExperimentRun` is
  one-run-one-condition by construction. No schema change.
- **English and Hinglish only.** That is exactly what RQ1 and RQ2 ask.
  Adding more languages would make the code-mix index undefined for most
  arms and break prompt parity, and would need its own RQ.
- **Memes are off.** RQ3 is a separate grid with `meme_injection.enabled`
  as the manipulated variable. Mixing it in here would confound RQ1.
- **1 trial to start.** Run the full grid once, look at real output, then
  raise `TRIALS` to 5 for the full design.


## 10. The grid

Each topic gets its own stance anchors. An unanchored 1-to-7 scale means
something different on immigration than on cycle lanes, so the anchors
have to be topic-specific or the scales are not comparable.

**Seeds depend on (topic, trial) but NOT on language.** That is Fix M: the
English and Hinglish arms of the same cell draw identical neighbour
samples and identical initial stances, so the only thing that differs
between them is the language directive.


In [ ]:
TOPICS = {
    'politics': dict(
        topic='whether immigration levels should be reduced',
        low='immigration levels should not be reduced at all',
        high='immigration levels should be reduced substantially'),
    'social': dict(
        topic='whether social media does more harm than good to teenagers',
        low='social media is not harmful to teenagers',
        high='social media is seriously harmful to teenagers'),
    'economic': dict(
        topic='whether the national minimum wage should be raised',
        low='the minimum wage should not be raised',
        high='the minimum wage should be raised substantially'),
    # Control: deliberately low-stakes. If stances polarize here too, the
    # driver is the echo-chamber mechanism, not the topic's divisiveness.
    'control': dict(
        topic='whether cities should expand cycle lane networks',
        low='cities should not expand cycle lanes',
        high='cities should expand cycle lanes substantially'),
}

LANGUAGES = ['english', 'hinglish']   # RQ1's manipulated variable
TRIALS    = [1]                       # raise to [1,2,3,4,5] for the full design
BASE_SEED = 2026

M, N, K, ALPHA = 100, 5, 10, 0.5      # Ohagi's baseline, unchanged

def seed_for(topic_key, trial):
    """Fix M: identical across language arms of the same cell, so
    neighbour sampling and initial stances are held constant while only
    language varies."""
    return BASE_SEED + list(TOPICS).index(topic_key) * 100 + trial

GRID = [(t, lang, tr) for t in TOPICS for lang in LANGUAGES for tr in TRIALS]
print(f'{len(GRID)} runs: {len(TOPICS)} topics x {len(LANGUAGES)} languages x {len(TRIALS)} trial(s)')


## 11. Dry run: what this will cost, before it spends anything

Equivalent to `run --dry-run` in the CLI design. Read this before running
the next cell.


In [ ]:
import json, pathlib, yaml
from sandbox.config_loader import load_run_config

CONFIG_DIR = pathlib.Path('configs/campaign'); CONFIG_DIR.mkdir(parents=True, exist_ok=True)

def write_config(topic_key, lang, trial):
    spec = TOPICS[topic_key]
    run_id = f'{topic_key}_{lang}_t{trial}'
    cfg = {
        'run_id': run_id, 'rq_target': 'RQ1_RQ2',
        'topic': spec['topic'],
        'alpha': ALPHA, 'M': M, 'N': N, 'K': K,
        'trial_number': trial,
        'language_condition': lang,
        'model_backend_id': f'hosted_vllm/{MODEL}',
        'api_base': 'http://localhost:8000/v1',
        'stance_scale': [1, 2, 3, 4, 5, 6, 7],
        'stance_low_label': spec['low'], 'stance_high_label': spec['high'],
        'persona_pool_id': 'pool_20',
        'meme_injection': {'enabled': False},   # RQ3 is a separate grid
        'seed': seed_for(topic_key, trial),
        'temperature': 0.7,
        'rate_limits': {}, 'checkpoint_every_n_turns': 1,
    }
    p = CONFIG_DIR / f'{run_id}.yaml'
    p.write_text(yaml.safe_dump(cfg, sort_keys=False))
    return p

paths = [write_config(*g) for g in GRID]
for p in paths:  load_run_config(p)          # validates every config now, not mid-campaign

calls = len(GRID) * M * K
TOK   = 950                                   # ~800 prompt + ~150 completion
for rate in (1000, 2000, 3000):
    print(f'  at {rate:>5} tok/s aggregate -> {calls*TOK/rate/3600:5.1f} GPU-hours')
print()
print(f'{len(GRID)} runs, {calls:,} calls, ~{calls*TOK/1e6:.0f}M tokens, $0.00')
print('all configs validated. MEASURE the real rate from run 1 before trusting the above.')


## 12. Run the campaign

Safe to re-run. Completed runs are skipped, and a run interrupted mid-way
resumes from its checkpoint byte-identically (Fix G plus the Phase 4
per-turn RNG derivation).

If Colab drops the session: re-run the server cell, then re-run this one.
It picks up exactly where it stopped.


In [ ]:
import time, urllib.request
from sandbox.config_loader import load_run_config
from sandbox.cost_tracker import CostTracker
from sandbox.model_gateway import ModelGateway
from sandbox.rate_limiter import RateLimiter
from sandbox.simulation_orchestrator import build_orchestrator

RUNS = pathlib.Path(RUNS_DIR if 'RUNS_DIR' in dir() else 'runs')

def already_done(run_id):
    """status is written only when a run reaches the end (Phase 4 2c)."""
    p = RUNS / run_id / 'run_config.json'
    return p.exists() and json.loads(p.read_text()).get('status') == 'completed'

def server_alive():
    try:
        urllib.request.urlopen('http://localhost:8000/v1/models', timeout=3); return True
    except Exception:
        return False

assert server_alive(), 'vLLM is not responding. Re-run the server cell first.'

for i, path in enumerate(paths, 1):
    run = load_run_config(path)
    if already_done(run.run_id):
        print(f'[{i}/{len(paths)}] {run.run_id}: already complete, skipping'); continue
    if not server_alive():
        print('server went away. re-run the server cell, then re-run this cell.'); break

    print(f'[{i}/{len(paths)}] {run.run_id}  ({run.language_condition}, seed {run.seed})', flush=True)
    t0 = time.time()
    tracker = CostTracker(run)
    gateway = ModelGateway(run.model_backend_id, RateLimiter(run.rate_limits),
                           tracker, api_base=run.api_base,
                           # Both come off the run, so run_config.json records
                           # what actually produced the dataset. max_concurrency
                           # falls back to 32 to match vLLM's --max-num-seqs.
                           max_concurrency=run.max_concurrency or 32,
                           top_logprobs=run.probe_top_logprobs)
    await build_orchestrator(run, gateway, cost_tracker=tracker, runs_dir=RUNS).run()
    mins = (time.time() - t0) / 60
    print(f'      done in {mins:.1f} min  ({M*K/(mins*60):.1f} calls/s)', flush=True)

print('\ncampaign finished (or paused). re-run this cell to continue.')


### Note on `await` here

This cell uses top-level `await` rather than `run_simulation.py`, because a
notebook already has an event loop and `asyncio.run()` would raise inside
it. Colab supports top-level `await`, so calling the orchestrator directly
is the correct form here. The script is for terminals.


## 13. Combine every run into one DataFrame

`interactions.jsonl` does not carry `topic` or `language_condition`, since
those live once per run in `run_config.json`. This joins them on so you can
group by either.

This is a stand-in for Phase 5's `load_run_dataframe`, which is not built
yet. Deliberately kept to the shape that function will have, so it lifts
into `sandbox/analysis/loader.py` unchanged rather than becoming a second
implementation.


In [ ]:
import pandas as pd

def load_run_dataframe(run_dir):
    """One run's interactions, with its run-level config joined on."""
    cfg = json.loads((run_dir / 'run_config.json').read_text())
    df = pd.read_json(run_dir / 'interactions.jsonl', lines=True)
    for col in ('topic', 'language_condition', 'trial_number', 'seed', 'run_id', 'alpha'):
        df[col] = cfg[col]
    df['topic_key'] = run_dir.name.split('_')[0]
    return df

completed = [d for d in sorted(RUNS.iterdir())
             if (d / 'run_config.json').exists() and already_done(d.name)]
df = pd.concat([load_run_dataframe(d) for d in completed], ignore_index=True)

out = RUNS / 'combined_interactions.parquet'
df.to_parquet(out)
print(f'{len(completed)} runs -> {len(df):,} interactions -> {out}')
df.head(3)


## 14. Did stance actually change?

The first question worth asking of real data, before any modelling.


In [ ]:
df['shift'] = (df.stance_after - df.stance_before).abs()
valid = df[df.api_call_status != 'failed_logged_null']

print('mean absolute stance shift per agent-turn\n')
print(valid.pivot_table(index='topic_key', columns='language_condition',
                        values='shift', aggfunc='mean').round(3))
print('\nfinal-turn stance spread (higher = more polarized)\n')
last = valid[valid.turn == valid.turn.max()]
print(last.pivot_table(index='topic_key', columns='language_condition',
                       values='stance_after', aggfunc='std').round(3))
print('\nparse-failure rate by language (an uneven rate is itself a confound)\n')
print(df.groupby('language_condition')
        .apply(lambda g: (g.api_call_status == 'failed_logged_null').mean(), include_groups=False)
        .round(4))
print('\nmean prompt tokens by language (Fix A truncation-symmetry check)\n')
print(valid.groupby('language_condition').prompt_token_count.mean().round(1))


## 15. Back up off Colab

Drive already persists this, so the zip is a second copy you can pull down
and keep with the paper. Cheap insurance against a Drive sync problem.


In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive('/content/chorus_runs', 'zip', RUNS)
print(f'{archive}  ({pathlib.Path(archive).stat().st_size/1e6:.1f} MB)')
files.download(archive)


## Scaling up from here

1. Check the tables in cell 14. Stances should move, and the parse-failure
   rate should be similar across languages. A much higher Hinglish failure
   rate means the model struggles with the format in that condition, which
   is a finding about the model, and it biases RQ1 if left unaddressed.
2. Read a few `reason_text` values from each language arm by hand. This is
   the A.10 Hinglish gate: is the Hinglish genuinely code-mixed, or English
   with a few Hindi words? If the latter, RQ1 is measuring nothing.
3. Set `TRIALS = [1,2,3,4,5]` and re-run cell 12. Completed runs are
   skipped, so only the 32 new ones execute.
4. RQ3 is a separate grid: same topics, `language_condition` fixed, and
   `meme_injection.enabled` toggled across runs.


---
# Part 3: RQ1

Parts 1 and 2 ran the generated regime: agents write their own posts and
report a stance as a number. RQ1 cannot use that. An English agent and a
Hindi agent asked to argue the same side produce different arguments, so any
difference in how far their readers move is a difference in content as much
as in language -- and an agent shown its own position as a digit can answer
by arithmetic on the numbers it was shown rather than by reading.

So RQ1 runs the **bank regime**: posts come from a fixed, pre-translated
argument CSV, and stance is read by a separate probe call whose next-token
logprobs over the digits 1-7 give a continuous value. Four language arms post
the *same argument*, differing in exactly three things -- the bank text, the
probe question, and one directive line.

Everything below writes under the Drive `RUNS_DIR` from cell 4, and every
step is resumable: re-running a cell skips what already finished.

**The order matters.** Each step consumes what the one before it wrote:

| step | writes | why it comes first |
|---|---|---|
| `make_rq1_configs.py` | `configs/rq1/campaign/` | everything else reads these |
| `check_comprehension.py` | `comprehension/` | drops items no arm can read |
| `calibrate_tau.py` | `calibration/tau.json` | every later "flip" is judged against it |
| `run_e1.py` | `e1/trials.jsonl` | the single-exposure effect |
| `run_campaign.py` | one directory per run | the networks |
| `analysis.metrics` | `results/*.csv` | the tidy tables |
| `analysis.stats` | the contrasts | the inference |
| `analysis.figures` | `results/figures/` | the pictures |

**The server must be running.** Re-run cell 12 if you restarted the session
since Part 2; everything here calls the same vLLM endpoint.

## 16. Generate the campaign configs

One base config expands into E3 (`delivery: on`) and E4 (`delivery: off`) for
each language and each seed index. **The seed is shared by all four languages
and by both experiments of one seed index** -- that is what makes a matched
set: `SeedManager` takes the base seed and nothing else, so the four arms get
the same graph, the same neighbour draws, and the same bank item for the same
sender on the same side.

Edit `configs/rq1/base.yaml` first. Every value marked PLACEHOLDER is a
smoke-test value: `M`, `N`, `K`, `graph_degree`, the bank paths, the persona
pool, the model and `max_concurrency` all need setting for a real campaign.

In [ ]:
RUNS = RUNS_DIR if 'RUNS_DIR' in dir() else 'runs'

# Check what the base config says before expanding it 40 times.
!sed -n '/^run_id/,$p' configs/rq1/base.yaml | grep -E 'PLACEHOLDER|^(M|N|K|model_backend_id|api_base|bank_path|persona_pool_id):'

In [ ]:
!python scripts/make_rq1_configs.py --seeds 5 --base-seed 1000 --out configs/rq1/campaign

import pandas as pd
manifest = pd.read_csv('configs/rq1/campaign/manifest.csv')
print()
print(manifest.head(10).to_string(index=False))
print(f'\n{len(manifest)} configs; seeds shared within a seed index:')
print(manifest[manifest.experiment != 'single'].groupby('seed_index').seed.nunique())

## 17. Smoke the campaign: two runs

`--max-runs 2` runs the first two configs only, which is enough to prove the
whole path works: the graph, the bank, the round-0 probe, both phases of a
round, the logprob probe and the checkpoint. Two runs is a cut inside a
matched set, so this is a plumbing check and not data -- the runner says so.

Delete `$RUNS/e3_english_ws_s01` and `$RUNS/e3_hinglish_ws_s01` before the
real campaign if you changed anything in `base.yaml` after this, since a
completed run is skipped rather than re-run.

In [ ]:
!python scripts/run_campaign.py configs/rq1/campaign --runs-dir "$RUNS" --max-runs 2

In [ ]:
# What one bank run actually produced. Read this before spending on the rest.
import pandas as pd, json, pathlib

run_dir = pathlib.Path(RUNS) / 'e3_english_ws_s01'
df = pd.read_json(run_dir / 'interactions.jsonl', lines=True)
print(df[['turn', 'speaker_agent_id', 'probe_mean', 'probe_mass',
          'items_read', 'api_call_status']].head(12).to_string(index=False))
print()
print('failed rows          ', (df.api_call_status == 'failed_logged_null').sum(), 'of', len(df))
print('probe mass (min/mean)', round(df.probe_mass.min(), 3), round(df.probe_mass.mean(), 3))
print('probe_mean spread    ', round(df.probe_mean.std(), 3))

# The audit the whole regime turns on: no position number reached an agent.
anchors = ('Answer with a single digit from 1 to 7 and nothing else.',
           'On this scale, 1 means', 'and 7 means')
leaks = 0
for line in (run_dir / 'prompts.jsonl').read_text(encoding='utf-8').splitlines():
    body = json.loads(line)['prompt_text']
    for phrase in anchors:
        body = body.replace(phrase, '')
    leaks += any(ch.isdigit() for ch in body)
print('prompts with a stray digit', leaks, '(must be 0; the scale\'s own digits are excluded)')

### Reading that

- **`probe_mass` near 1.0** means the model answered on the scale. Below the
  configured floor (0.5) the read is refused and the row is logged as failed --
  never renormalised, because a stance manufactured out of scraps looks
  exactly like a real one in the output.
- **`probe_mean` with no spread at all** means every agent answered the same
  digit, which usually means the probe question is not landing. Read it in
  `base.yaml` and try it by hand against the server.
- **A stray digit in a prompt** is a bug, not a tuning problem. Stop and fix
  it: an agent that can see a position number does not have to read the
  arguments.

## 18. Comprehension check

Can the model read each bank item, in each language, at all? Without this,
"the Hindi rendering moved readers less" and "the model cannot read the Hindi
rendering" are the same number.

Each item is shown with no persona and no lean, and the model is asked which
of the two views it supports -- in English, in both label orders, so a
standing preference for "A" cannot produce a pass. An item passes only when
the correct letter carries more than half the probability in **both** orders.

Items failing in **any** language go to `comprehension/failed_items.txt`,
which is a valid `exclude_items_path` exactly as written. Any rather than all:
the bank holds one argument rendered four ways, so keeping it in the arms
that could read it would make the arms differ in which arguments they
contain.

In [ ]:
!python scripts/check_comprehension.py --runs-dir "$RUNS" --configs configs/rq1/campaign

In [ ]:
import pandas as pd, pathlib

summary = pd.read_csv(pathlib.Path(RUNS) / 'comprehension' / 'summary.csv')
print(summary.pivot_table(index='item_id', columns='language',
                          values='p_correct_high_first').round(3).to_string())
print()
failed = pathlib.Path(RUNS) / 'comprehension' / 'failed_items.txt'
print(failed.read_text())
print('To drop them, set in configs/rq1/base.yaml and regenerate:')
print(f'  exclude_items_path: {failed}')

## 19. Calibrate τ

A probe reading moves on its own: the history grows, the sampler draws. τ is
how much, measured per language by showing a reader one side-free message
from the E4 control's own neutral bank, asking it to reply, and probing again.
`τ` is the 95th percentile of |Δx| over those trials and `τ_conf` the same of
|Δ confidence|.

Every later claim -- that an agent flipped, hardened or softened -- is a claim
that a change exceeded this. **The noise is not the same size in all four
arms**, so one threshold for the campaign would be wrong in three of them.

The default sweep is every persona × all seven leans × every neutral message,
per language. `--leans 1 4 7` cuts it to three starting positions, which still
clears the 30-trials-per-language floor with the 20-persona pool and spends
about 40% of the calls.

In [ ]:
!python scripts/calibrate_tau.py --runs-dir "$RUNS" --configs configs/rq1/campaign --leans 1 4 7

In [ ]:
import json, pathlib

tau = json.loads((pathlib.Path(RUNS) / 'calibration' / 'tau.json').read_text())
print(f"percentile {tau['percentile']}   commit {tau['git_commit_hash'][:8]}   smoke={tau['smoke']}")
for language, arm in tau['languages'].items():
    print(f"  {language:<16} tau={arm['tau']:.4f}  tau_conf={arm['tau_conf']:.4f}  "
          f"n_ok={arm['n_ok']:<5} n_failed={arm['n_failed']:<4} "
          f"min mass={min(arm['mass0_min'], arm['mass1_min']):.3f}")
print()
print('A tau far larger in one arm than the others means an equal amount of real')
print('movement is called a flip in one arm and nothing in another. Check its')
print('probe mass before reading any result.')

## 20. E1: one argument, one reader

RQ1's primary question in the smallest setting that can answer it. A reader is
probed, shown one bank item, and probed again. The argument is held identical
across the four arms, so a difference in how far readers move is a difference
in the rendering and nothing else.

**No reply call by default** -- the reader only reads. A seeded reply is a
second thing happening between the two readings and is where most of a
trial's variance comes from, so `--with-reply` measures that version into its
own folder instead of mixing the two in one column.

`toward` signs the movement by the item's side, so "moved the way it argued"
is positive whichever side that was.

In [ ]:
!python scripts/run_e1.py --runs-dir "$RUNS" --configs configs/rq1/campaign --leans 2 4 6

## 21. Run the campaign

Order: seed index, then experiment, then language. A whole matched set --
four languages of E3, then four of E4, all sharing one seed -- finishes before
the next seed index starts, so a session that dies leaves **complete matched
sets** rather than three arms of a four-arm comparison.

Safe to re-run. A completed run is skipped and an interrupted one resumes from
its checkpoint, which is why `--runs-dir` must point at Drive: the Colab
filesystem is ephemeral and the checkpoint is what resume depends on.

Drop `--max-runs` for the full 40. Measure the first matched set's wall time
before trusting any estimate of the rest.

In [ ]:
!python scripts/run_campaign.py configs/rq1/campaign --runs-dir "$RUNS"

## 22. Metrics

Each run becomes three tidy tables. Every round-to-round change is classified
against that run's own language τ:

- **flip** -- the sign of x changed and |Δx| > τ
- **harden / soften** -- same sign, |x| grew or shrank by more than τ
- **conf_up / conf_down** -- did not move, but became more or less decided
- **none** -- otherwise

`x == 0` is its own sign, so a move off the midpoint is a change of sign.

`t_stationary` is the first round after which var(x), extremity and
assortativity each changed by less than their ε for three consecutive rounds.
`--epsilon-from-e4` takes each ε from the control arm's own round-to-round
wobble, which is the population-level analogue of what τ is for one agent.

In [ ]:
!python -m analysis.metrics \
    --runs-dir "$RUNS" \
    --tau "$RUNS/calibration/tau.json" \
    --out "$RUNS/results" \
    --epsilon-from-e4

In [ ]:
import pandas as pd, pathlib

results = pathlib.Path(RUNS) / 'results'
runs = pd.read_csv(results / 'run_metrics.csv')
print(runs.groupby(['experiment', 'language'])[
    ['final_var_x', 'final_extremity', 'final_assortativity',
     't_stationary', 'rate_flip', 'mean_n_repeat_reads']
].mean().round(4).to_string())
print()
print('n_failed over the whole campaign:', int(runs.n_failed_total.sum()),
      'of', int(runs.n_valid_total.sum() + runs.n_failed_total.sum()))
print('runs that never reached stationarity:', int((~runs.stationary).sum()), 'of', len(runs))

## 23. Statistics

Two analyses with two different units.

**E1 is trial-level**: a mixed model with a random intercept per item (some
arguments are simply more persuasive) and a variance component for persona.
The manipulation check is reported first -- if a strong argument does not move
an English reader further than a weak one, the bank's own strength labels do
not describe the stimuli and no language comparison over them means anything.
A non-converged fit is never reported as a result; the fallback is
cluster-robust OLS, and `e1_model.txt` leads with which one you have.

**The networks are run-level, paired by seed.** A hundred agents within one
run are not a hundred independent observations. The primary comparison is the
difference-in-differences: a raw E3 difference mixes what the language did to
persuasion with what it did to the measurement, and the E4 control is exposed
to the second and not the first.

With fewer than six seeds the smallest attainable two-sided Wilcoxon p value
is above 0.05, so no row could be significant however large the effect. The
script says so rather than leaving you to notice.

In [ ]:
!python -m analysis.stats \
    --e1 "$RUNS/e1/trials.jsonl" \
    --results "$RUNS/results" \
    --out "$RUNS/results"

In [ ]:
import pandas as pd, pathlib

results = pathlib.Path(RUNS) / 'results'
print((results / 'e1_model.txt').read_text().split('=' * 72)[0])

contrasts = pd.read_csv(results / 'e1_contrasts.csv')
print(contrasts[['family', 'contrast', 'estimate', 'ci_low', 'ci_high',
                 'p_raw', 'p_holm', 'reject_at_05']].round(4).to_string(index=False))
print()
network = pd.read_csv(results / 'network_contrasts.csv')
print(network[network.scope == 'did'][
    ['metric', 'language', 'n', 'mean_diff', 'ci_low', 'ci_high',
     'p_holm', 'reject_at_05', 'underpowered']
].round(4).to_string(index=False))

## 24. Figures

Four figures as PDF and PNG. One colour per language, used in all four; E3 and
E4 are solid and dashed, strength is position. The PDFs are the ones for the
paper.

In [ ]:
!python -m analysis.figures \
    --results "$RUNS/results" \
    --e1 "$RUNS/e1/trials.jsonl" \
    --calibration "$RUNS/calibration" \
    --out "$RUNS/results/figures"

In [ ]:
from IPython.display import Image, display
import pathlib

figures = pathlib.Path(RUNS) / 'results' / 'figures'
for stem in ('tau', 'e1_toward', 'trajectories', 'instrument'):
    print(stem)
    display(Image(filename=str(figures / f'{stem}.png')))

## 25. What to check before believing any of it

1. **The manipulation check** (cell 23). A strong argument must move an
   English reader further than a weak one. If it does not, the bank's own
   labels do not describe the stimuli.
2. **The comprehension pass rate** (cell 18). An arm that cannot read its own
   items will show a small effect for a reason that has nothing to do with
   persuasion.
3. **τ across arms** (cell 19). Wildly different thresholds mean the four arms
   are not equally precise instruments, which is a result worth reporting on
   its own.
4. **Probe mass and prompt length by arm** (the `instrument` figure). The bank
   regime reduces the prompt-length gap and does not remove it -- the same
   argument in Hinglish still tokenizes to more than in English, and with full
   history that gap compounds over a run. Reporting it is part of the result.
5. **`n`** in every network row. Five seeds cannot produce a significant
   signed-rank result. Run more before reading the p values.